# Notebook 9 — Feature Selection
Now that Notebooks 2-8 produced a wide engineered feature set, this notebook answers
the central question: **which features actually earn their place in the model?**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, chi2, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])

# Rebuild a compact engineered feature set (condensed from Notebooks 2-7) for selection demo
ref = pd.Timestamp("2024-06-30")
customers["tenure_days"] = (ref - customers["signup_date"]).dt.days
customers["total_charges"] = customers["total_charges"].fillna(customers["monthly_charges"])
customers["charge_per_tenure_month"] = customers["monthly_charges"] / customers["tenure_months"].replace(0,1)
customers["log_total_charges"] = np.log1p(customers["total_charges"])
customers["contract_ordinal"] = customers["contract"].map({"Month-to-month":0,"One year":1,"Two year":2})
customers["is_electronic_check"] = (customers["payment_method"]=="Electronic check").astype(int)
customers["is_fiber"] = (customers["internet_service"]=="Fiber optic").astype(int)
customers["senior_x_charges"] = customers["senior_citizen"] * customers["monthly_charges"]
customers["random_noise"] = np.random.default_rng(0).normal(0,1,len(customers))  # deliberately useless control feature
customers["constant_col"] = 1  # deliberately zero-variance control feature

customers["churn_binary"] = (customers["churn"]=="Yes").astype(int)

feature_cols = [
    "monthly_charges","tenure_months","tenure_days","total_charges","log_total_charges",
    "charge_per_tenure_month","contract_ordinal","is_electronic_check","is_fiber",
    "senior_x_charges","senior_citizen","random_noise","constant_col"
]
X = customers[feature_cols].fillna(0)
y = customers["churn_binary"]
X.shape, y.mean()

## 1. Why Feature Selection?

More features are **not** automatically better. Irrelevant or redundant features:
- Add noise that can hurt generalization, especially for models sensitive to
  dimensionality (linear models, KNN)
- Increase training/inference cost and pipeline maintenance burden
- Make the model harder to interpret and debug
- Can actively mislead feature importance analysis if highly correlated with a
  genuinely useful feature

**Relevant** features have a real relationship with the target. **Irrelevant**
features (like our deliberately-added `random_noise`) have none. **Redundant**
features duplicate information already captured by another feature (e.g.
`tenure_months` and `tenure_days` — same signal, different units).

## 2. Variance Threshold (Filter Method)

Removes features with little to no variance — they're constant (or nearly constant)
across all rows, so they cannot possibly help a model distinguish between classes.

In [ ]:
vt = VarianceThreshold(threshold=0.0)
vt.fit(X)
zero_variance_features = X.columns[~vt.get_support()].tolist()
print("Zero-variance features detected:", zero_variance_features)

Correctly flags our deliberately-inserted `constant_col`. This is the cheapest,
fastest first pass in any real feature selection pipeline.

## 3. Correlation-Based Selection

Removes one of each pair of features that are highly correlated with each other
(redundant), and separately checks each feature's correlation with the target
(relevance).

In [ ]:
corr_matrix = X.drop(columns=["constant_col"]).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_pairs = [(col, row, upper.loc[row,col]) for col in upper.columns for row in upper.index
                    if upper.loc[row,col] > 0.85]
print("Highly correlated (redundant) pairs (>0.85):")
for a,b,v in redundant_pairs:
    print(f"  {a} <-> {b}: {v:.3f}")

In [ ]:
target_corr = X.drop(columns=["constant_col"]).apply(lambda col: col.corr(y)).sort_values(key=abs, ascending=False)
target_corr

`random_noise` correctly shows near-zero correlation with `churn_binary` — confirming
our filter methods are working as expected before we trust them on real features.

## 4. Mutual Information (Filter Method — captures non-linear relationships)

Unlike Pearson correlation (linear only), Mutual Information captures **any**
statistical dependency, linear or not — important because many churn relationships
(e.g. `charge_per_tenure_month`) are non-linear.

In [ ]:
mi_scores = mutual_info_classif(X.drop(columns=["constant_col"]), y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.drop(columns=["constant_col"]).columns).sort_values(ascending=False)
mi_series

## 5. Chi-Square Test (for non-negative / categorical-like features)

Chi-Square measures dependence between non-negative features and a categorical target
— commonly used after encoding categorical variables. Requires non-negative inputs, so
we scale first.

In [ ]:
X_nonneg = MinMaxScaler().fit_transform(X.drop(columns=["constant_col","random_noise"]).clip(lower=0))
chi_scores, p_values = chi2(X_nonneg, y)
chi_df = pd.DataFrame({
    "feature": X.drop(columns=["constant_col","random_noise"]).columns,
    "chi2_score": chi_scores, "p_value": p_values
}).sort_values("chi2_score", ascending=False)
chi_df

## 6. Wrapper Method — Recursive Feature Elimination (RFE)

RFE repeatedly trains a model, drops the weakest feature, and retrains — directly
optimizing for the *model's* performance rather than a proxy statistic. More
computationally expensive than filter methods, but accounts for feature interactions.

In [ ]:
model = LogisticRegression(max_iter=1000)
rfe = RFE(model, n_features_to_select=6)
rfe.fit(X.drop(columns=["constant_col"]), y)

rfe_ranking = pd.Series(rfe.ranking_, index=X.drop(columns=["constant_col"]).columns).sort_values()
rfe_ranking  # rank 1 = selected

## 7. Embedded Method — L1 (Lasso) Regularization

L1 regularization shrinks unimportant feature coefficients **exactly to zero** as a
side-effect of model training itself — selection is "embedded" in the fitting process,
rather than a separate step.

In [ ]:
lasso_model = LogisticRegression(penalty="l1", solver="liblinear", C=0.3, max_iter=1000)
lasso_model.fit(X.drop(columns=["constant_col"]).apply(lambda c: (c-c.mean())/c.std()), y)

coef_series = pd.Series(lasso_model.coef_[0], index=X.drop(columns=["constant_col"]).columns)
coef_series.sort_values(key=abs, ascending=False)

## 8. Embedded Method — Tree-Based Feature Importance

Random Forest / Gradient Boosted Trees compute feature importance as a byproduct of
how much each feature reduces impurity across all trees — covered in depth in
Notebook 10.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, max_depth=6)
rf.fit(X.drop(columns=["constant_col"]), y)
rf_importance = pd.Series(rf.feature_importances_, index=X.drop(columns=["constant_col"]).columns).sort_values(ascending=False)
rf_importance

## Advantages & Limitations of Each Approach

| Method | Advantages | Limitations |
|---|---|---|
| Variance Threshold | Extremely cheap, no target needed | Only catches the most obvious junk features |
| Correlation-based | Simple, interpretable | Misses non-linear relationships |
| Mutual Information | Captures non-linear dependency | Needs enough data to estimate reliably; no interaction awareness |
| Chi-Square | Good for categorical/count features | Requires non-negative inputs; less suited to continuous data |
| RFE (wrapper) | Accounts for feature interactions, optimizes actual model performance | Computationally expensive; result tied to the chosen base model |
| L1/Lasso (embedded) | Selection built into training, efficient | Only captures linear relationships; sensitive to feature scaling |
| Tree-based importance (embedded) | Captures non-linear/interaction effects, fast | Biased toward high-cardinality features; correlated features can split importance |

## Final Decision — Consolidated Across Methods

`random_noise` and `constant_col` were **correctly rejected by every method** —
strong validation that our selection process works. Features consistently ranked
highly across multiple methods (`charge_per_tenure_month`, `contract_ordinal`,
`monthly_charges`, `is_electronic_check`, `is_fiber`) are retained with high
confidence for the final ML-ready dataset built in Notebook 15.